# BGDY 3x3 · ampliar el dataset con más amplitudes de rattle (σ = 0.05 – 0.10)

Continuación de `make_rattled.ipynb`: genera **4 tandas más de 40 configuraciones** (σ = 0.05, 0.06, 0.08 y 0.10 Å), es decir
160 casos nuevos que siguen a los que ya tienes (`case_201` … `case_360`).

- **No toca nada de lo existente**: solo crea carpetas nuevas en `cases/`.
- **Mismo `input_scf.in`**: si tienes `cases/case_001`, copia su cabecera; si no, usa la de `make_rattled.ipynb`.
- **Misma estructura de carpeta** (`input_scf.in` + `initial.xyz` con su `rattle_std`), así que `job_scf.sh`, `status.sh` y
  `collect.py` funcionan tal cual.

Ejecútalo en la misma carpeta que `make_rattled.ipynb` (con `relax_final.xyz`). Necesita `ase` y `hiphive`.

## 1. Parámetros

**Ojo con estas amplitudes.** En hiPhive el desplazamiento típico es ≈ √10 · σ por componente, así que con σ = 0.05 – 0.10
el desplazamiento rms es de unos **0.3 – 0.5 Å** y algún átomo se mueve más de 1 Å (casi una longitud de enlace). El filtro de
distancia mínima solo impide que los átomos choquen: son estructuras **muy deformadas**. Espera energías altas, SCF más lentos y
algún caso sin converger (mira el README para eso). Por eso al final se proponen primero los dos extremos.

In [ ]:
import numpy as np
from pathlib import Path
from ase.io import read, write
from ase.neighborlist import neighbor_list
from hiphive.structure_generation import generate_mc_rattled_structures

REF  = "relax_final.xyz"           # lámina 3x3 relajada (la misma de siempre)
STDS = [0.05, 0.06, 0.08, 0.10]    # amplitudes nuevas (Å): una tanda por valor
N    = 40                          # configuraciones por amplitud  →  4 × 40 = 160
SEED = 1042                        # distinta de la tanda original (42): así no se repiten los patrones de desplazamiento

D_MIN_FACTOR    = 0.93             # igual que en make_rattled.ipynb
MIN_DIST_FACTOR = 0.90

## 2. Continuar donde lo dejaste

La numeración sigue al último `case_XXX` que haya en `cases/`. Si `cases/` no está en esta carpeta (p. ej. solo lo tienes en el
cluster), se asume que ya tienes 200 y los nuevos empiezan en `case_201`. Si algún σ de `STDS` ya está en `cases/`, se para: así
no se duplica una tanda por error.

In [ ]:
OUT = Path("cases")
previos = sorted(OUT.glob("case_*"))
PRIMER = max((int(d.name[5:]) for d in previos), default=200) + 1       # primer número libre

ya = {read(d / "initial.xyz").info["rattle_std"] for d in previos}      # σ que ya tienes
assert not ya & set(STDS), f"ya tienes casos con rattle_std = {sorted(ya & set(STDS))}"
print(f"{len(previos)} casos previos · los nuevos empiezan en case_{PRIMER:03d}")

## 3. Input de QE

La misma cabecera de `make_rattled.ipynb` (la celda de abajo es una copia). Si existe `cases/case_001/input_scf.in` se usa
**su** cabecera en lugar de la copia: así los casos nuevos son idénticos a los antiguos aunque hayas retocado algún parámetro
(`mixing_beta`, `conv_thr`…).

In [ ]:
HEAD = """&CONTROL
 calculation = 'scf'
 prefix = 'bgdy'
 pseudo_dir = '../../pseudos'
 outdir = './save'
 disk_io = 'nowf'
 tprnfor = .true.
 tstress = .true.
/
&SYSTEM
 ibrav = 0
 nat = {nat}
 ntyp = 2
 nspin = 2
 starting_magnetization(1) = 0.1d0
 starting_magnetization(2) = 0.1d0
 ecutwfc = 60.0
 ecutrho = 600.0
 vdw_corr = 'Grimme-D3'
 occupations = 'smearing'
 smearing = 'gaussian'
 degauss = 0.005
 nosym = .true.
/
&ELECTRONS
 electron_maxstep = 250
 mixing_mode = 'plain'
 mixing_beta = 0.2
 conv_thr = 7.35d-6
/
ATOMIC_SPECIES
C 12.011 C.pbe-n-kjpaw_psl.1.0.0.UPF
B 10.811 B.pbe-n-kjpaw_psl.1.0.0.UPF
"""

primero = OUT / "case_001" / "input_scf.in"
if primero.exists():                                          # cabecera de tus casos (ya con nat rellenado)
    HEAD = primero.read_text().split("CELL_PARAMETERS")[0]
    print("cabecera copiada de", primero)

## 4. Generar el rattle

Exactamente el procedimiento de `make_rattled.ipynb`: `generate_mc_rattled_structures` con `d_min = 0.93 · d0` y un filtro duro
que descarta toda configuración con algún par a menos de `0.90 · d0`, reponiendo hasta completar `N`.

Con σ grande el filtro descarta **la mayor parte** de lo generado (en mis pruebas con tu lámina, entre el 65 y el 80 %); el
último número de cada línea te lo dice. Compruébalos: `d_min` ≥ 1.11 Å y los desplazamientos rms / máximo.

In [ ]:
ref = read(REF)

def dmin(atoms):
    """Distancia mínima entre pares de átomos (con condiciones periódicas)."""
    return neighbor_list("d", atoms, 3.0).min()

d0 = dmin(ref)
configs = []                                           # lista de (rattle_std, Atoms)

for k, std in enumerate(STDS):
    good, generadas = [], 0
    for t in range(50):                                # repone las descartadas con otra semilla
        batch = generate_mc_rattled_structures(ref, N, std, D_MIN_FACTOR * d0, seed=SEED + 100 * k + t)
        generadas += len(batch)
        good += [a for a in batch if dmin(a) >= MIN_DIST_FACTOR * d0]
        if len(good) >= N:
            break
    else:
        raise RuntimeError(f"rattle_std={std}: no salen {N} configuraciones válidas; baja la amplitud")
    good = good[:N]

    u = np.array([a.positions - ref.positions for a in good])      # desplazamientos, forma (N, 126, 3)
    print(f"std={std:.3f}  |u| rms={np.sqrt((u**2).sum(2).mean()):.3f} Å  "
          f"max={np.linalg.norm(u, axis=2).max():.3f} Å  d_min={min(dmin(a) for a in good):.3f} Å  "
          f"(válidas {N} de {generadas} generadas)")
    configs += [(std, a) for a in good]

print(f"\n{len(configs)} configuraciones nuevas")

## 5. Escribir las carpetas

Una carpeta por configuración, igual que antes: `input_scf.in` (cabecera + `CELL_PARAMETERS` + `ATOMIC_POSITIONS` +
`K_POINTS gamma`) e `initial.xyz` con su `rattle_std`. `mkdir` falla si la carpeta ya existe, así que **nunca se sobrescribe** un caso.

In [ ]:
for i, (std, a) in enumerate(configs, start=PRIMER):
    d = OUT / f"case_{i:03d}"
    d.mkdir(parents=True)

    a.info["rattle_std"] = std
    write(d / "initial.xyz", a)

    cell = "".join("%.10f %.10f %.10f\n" % tuple(v) for v in a.cell)
    pos = "".join("%-2s %.10f %.10f %.10f\n" % (s, *p) for s, p in zip(a.symbols, a.positions))
    (d / "input_scf.in").write_text(
        HEAD.format(nat=len(a)) + f"CELL_PARAMETERS angstrom\n{cell}ATOMIC_POSITIONS angstrom\n{pos}K_POINTS gamma\n")

ult = PRIMER + len(configs) - 1
print(f"{len(configs)} carpetas nuevas: case_{PRIMER:03d} … case_{ult:03d}\n")
print("En el cluster, primero los dos extremos (σ menor y mayor) para ver tiempo y convergencia:")
print(f"  sbatch --array={PRIMER},{ult} job_scf.sh")
print("y si van bien, el resto, máx. 20 a la vez:")
print(f"  sbatch --array={PRIMER + 1}-{ult - 1}%20 job_scf.sh")

## 6. Siguiente paso

Copia al cluster **solo las carpetas nuevas** (`cases/case_201` … `cases/case_360`); `job_scf.sh` no cambia, el `--array` ya
dice qué casos lanzar. Para σ altos los SCF tardan más: si un job se corta por tiempo, relánzalo con más margen
(`sbatch --time=05:00:00 --array=<lista> job_scf.sh`) y, si no converge, baja `mixing_beta` como indica el README.

`status.sh` y `collect.py` recorren todos los `cases/case_*`, así que `dataset.extxyz` saldrá con las 360 configuraciones y en
`analisis_rattle.ipynb` los σ nuevos aparecen solos.